# Lab 5: Build the knowledge layer (RAG ingestion)
**Module 3.1: RAG fundamentals.**  **Time:** 30 minutes.

**Goal:** take 10 policy PDFs from a fictional company, Meridian Retail, and make them searchable by meaning:
**load, chunk, embed, store, search.**

Vector database: **Qdrant in local mode**. It runs inside this notebook, no account or server needed.
In production you change one line to point the same code at a Qdrant server or Qdrant Cloud.

In [ ]:
# SETUP: run this cell first. It works in Google Colab and in VS Code.
import os, sys, subprocess
REPO_URL = "https://github.com/cybertide-solutions/ai-platform-architect-labs.git"
if "google.colab" in sys.modules:
    if not os.path.exists("/content/course"):
        subprocess.run(["git", "clone", "-q", REPO_URL, "/content/course"], check=True)
    os.chdir("/content/course")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")                      # VS Code starts in notebooks/; the course files are one level up
sys.path.insert(0, os.getcwd())
import labkit as lk
lk.setup();

## 1. Load
Each PDF page becomes one record with its text and metadata (document, page, department).

In [ ]:
pages = lk.load_pdfs()
print(len(pages), "pages from", len({p["doc"] for p in pages}), "documents")
print(pages[8]["doc"], "page", pages[8]["page"], "department:", pages[8]["dept"])
print(pages[8]["text"][:500])

## 2. Chunk
Pages are split into chunks of about 600 characters with 100 characters of overlap, so a sentence cut at a
boundary still appears whole in one of the neighbours.

In [ ]:
chunks = lk.chunk_pages(pages, size=600, overlap=100)
print(len(chunks), "chunks")
import pandas as pd
pd.Series([len(c["text"]) for c in chunks]).describe().round(0)

In [ ]:
print(chunks[3]["text"], "\n---\n", chunks[4]["text"])

## 3. Embed
An embedding model turns text into a vector (a list of numbers). Texts with similar meaning get similar
vectors. We use a small open model (`model2vec`) that runs on any laptop CPU.

In [ ]:
v = lk.embed(["How many leave days do I get?"])
print("vector length:", v.shape[1], "  first 5 numbers:", v[0][:5].round(3), "  backend:", lk.EMBED_INFO)

In [ ]:
import numpy as np
sentences = ["How many holidays do I get per year?",
             "Annual leave entitlement is 24 days.",
             "Laptops are replaced every 4 years."]
E = lk.embed(sentences)
sim = pd.DataFrame(E @ E.T, index=sentences, columns=["holidays?", "leave 24d", "laptops"]).round(2)
sim  # cosine similarity: closer to 1 means closer in meaning

## 4. Store in Qdrant
A **collection** holds the vectors plus a **payload** (metadata) for each chunk.

In [ ]:
from qdrant_client import QdrantClient, models
qc = QdrantClient(":memory:")          # production: QdrantClient(url="https://...", api_key="...")
qc.create_collection("policies", vectors_config=models.VectorParams(size=v.shape[1], distance=models.Distance.COSINE))

vectors = lk.embed([c["text"] for c in chunks])
qc.upsert("policies", points=[
    models.PointStruct(id=c["id"], vector=vec.tolist(),
                       payload={"doc": c["doc"], "page": c["page"], "dept": c["dept"], "text": c["text"]})
    for c, vec in zip(chunks, vectors)])
print("points stored:", qc.count("policies").count)

## 5. Search by meaning
Notice that the query words do not need to match the document words.

In [ ]:
def show(query, dept=None, k=3):
    print(f"\nQUERY: {query}" + (f"   (only {dept})" if dept else ""))
    for h in lk.search(qc, query, k=k, dept=dept):
        print(f"  {h['score']:.3f}  {h['doc']} p.{h['page']}  {h['text'][:80]!r}")

show("Can I work from home on Fridays?")
show("What happens if I lose my laptop?")
show("How much can I spend on a hotel in Bengaluru?")

## 6. Metadata filters
Filters run inside the vector database. They are how you enforce **access control** in RAG:
a user only retrieves chunks from departments they may see.

In [ ]:
show("What are the approval limits?")
show("What are the approval limits?", dept="Finance")
show("What are the approval limits?", dept="Customer Service")

## Try this
1. Re-run the chunking with `size=300` and with `size=1500`. How many chunks do you get? Look at one chunk.
2. Ask a question in Hindi or Hinglish, e.g. `"chhutti kitne din ki milti hai?"`. Does retrieval still work?
   (This tells you whether this embedding model fits your users.)

**Discuss:** where should access-control metadata come from in a real company (document management system,
Active Directory groups)? What happens when a document's permissions change after it was indexed?